In [2]:

 from google.colab import drive
 drive.mount('/content/drive')
 from google.colab import files
 uploaded = files.upload()  # subir oulad_kongo.xlsx

!pip install -q scikit-learn seaborn pandas numpy matplotlib openpyxl


Mounted at /content/drive


Saving AnonymisezData_oulad_context-Kongo-2024 (2).xlsx to AnonymisezData_oulad_context-Kongo-2024 (2).xlsx


In [3]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings("ignore")

from collections import namedtuple
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

RANDOM_STATE = 42
DATA_PATH = "/content/AnonymisezData_oulad_context-Kongo-2024 (2).xlsx"  # ajustar ruta según entorno (Colab/local)


---
## 1. OBTAIN — Obtención de datos




In [4]:
StudentRecord = namedtuple(
    "StudentRecord",
    ["student_id", "code_module", "code_presentation", "weighted_score",
     "total_clicks", "active_weeks", "risk_level"]
)  # TAD: registro inmutable de estudiante procesado


class OULADDataLoader:
    """Encapsula la obtención de datos crudos desde el libro Excel OULAD-Kongo."""

    SHEETS = ["StudentInfo", "Registration", "Assesss_detail", "Assess Plan",
              "VLE_clickStream", "Vle_modules", "cursos"]

    def __init__(self, path: str = DATA_PATH):
        self._path = path
        self._tables: Dict[str, pd.DataFrame] = {}

    def load(self) -> "OULADDataLoader":
        xls = pd.ExcelFile(self._path)
        for sheet in self.SHEETS:
            self._tables[sheet] = pd.read_excel(xls, sheet_name=sheet)
        return self

    def get(self, sheet: str) -> pd.DataFrame:
        return self._tables[sheet].copy()

    @property
    def tables(self) -> Dict[str, pd.DataFrame]:
        return {k: v.copy() for k, v in self._tables.items()}


loader = OULADDataLoader().load()
for name, df in loader.tables.items():
    print(f"{name:20s} -> {df.shape}")


StudentInfo          -> (148, 12)
Registration         -> (148, 5)
Assesss_detail       -> (822, 22)
Assess Plan          -> (28, 6)
VLE_clickStream      -> (3950, 13)
Vle_modules          -> (57, 6)
cursos               -> (2, 3)


---
## 2. SCRUB — Limpieza, missing values y ajustes ordinales


In [5]:
class DataCleaner:
    """Maneja valores faltantes y ajustes de tipo/ordinalidad."""

    IMD_ORDER = ["0-10%", "10-20%", "20-30%", "30-40%", "40-50%",
                 "50-60%", "60-70%", "70-80%", "80-90%", "90-100%"]
    AGE_ORDER = ["18-20", "20-25", "25-30", "30-35", " > 35"]

    def clean_student_info(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df["imputed_demo"] = df["imd_band"].isna().astype(int)

        df["imd_band"] = df["imd_band"].fillna(df["imd_band"].mode(dropna=True)[0])
        df["age_band"] = df["age_band"].fillna(df["age_band"].mode(dropna=True)[0])
        df["num_of_prev_attempts"] = df["num_of_prev_attempts"].fillna(
            df["num_of_prev_attempts"].median())
        df["studied_credits"] = df["studied_credits"].fillna(
            df["studied_credits"].median())

        df["imd_band_ord"] = df["imd_band"].apply(
            lambda x: self.IMD_ORDER.index(x) if x in self.IMD_ORDER else np.nan)
        df["age_band_ord"] = df["age_band"].apply(
            lambda x: self.AGE_ORDER.index(x) if x in self.AGE_ORDER else np.nan)
        return df

    def clean_assess_detail(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df["score"] = df["score"].fillna(0.0)
        return df

    def clean_vle(self, df: pd.DataFrame, valid_ids: set) -> pd.DataFrame:
        df = df.copy()
        df = df[df["guid_student_id"].isin(valid_ids)]
        df["sum_clics"] = df["sum_clics"].fillna(0)
        return df


cleaner = DataCleaner()
si_clean = cleaner.clean_student_info(loader.get("StudentInfo"))
ad_clean = cleaner.clean_assess_detail(loader.get("Assesss_detail"))
vle_clean = cleaner.clean_vle(loader.get("VLE_clickStream"), set(si_clean["guid_student_id"]))

print("Registros imputados:", si_clean["imputed_demo"].sum(), "de", len(si_clean))
si_clean.isna().sum()


Registros imputados: 18 de 148


,0
code_module,0
code_presentation,0
guid_student_id,0
gender,0
region,0
highest_education,0
imd_band,0
age_band,0
num_of_prev_attempts,0
studied_credits,0


---
## 3. EXPLORE — EDA de alto nivel

Se examinan distribuciones univariadas, dispersión bivariada, matriz de
correlación, asimetría y curtosis, y boxplots para detección de outliers.


In [6]:
ad_exam = ad_clean[ad_clean["assessment_type"] == "EXAM"]
print("Estudiantes con al menos un examen calificado:", ad_exam["guid_student_id"].nunique(), "/", si_clean.shape[0])
print(ad_clean["assessment_type"].value_counts())
print(ad_clean["status"].value_counts())


Estudiantes con al menos un examen calificado: 87 / 148
assessment_type
TMA     611
EXAM    211
Name: count, dtype: int64
status
new           611
finished      180
inprogress     31
Name: count, dtype: int64


In [7]:
# Estadística descriptiva preliminar (antes de ensamblar features finales)
si_clean[["num_of_prev_attempts", "studied_credits"]].describe()


,num_of_prev_attempts,studied_credits
count,148.000000,148.000000
mean,2.445946,20.270270
std,1.826799,26.615068
min,1.000000,0.000000
25%,1.000000,0.000000
50%,2.000000,14.000000
75%,3.000000,25.000000
max,10.000000,148.000000


## 4. Ingeniería de variables y construcción de hipótesis (Feature Engineering)


In [ ]:
class FeatureEngineer:
    """
    H1: el nivel de interacción (clics) en el VLE está asociado
        positivamente con el desempeño académico ponderado.
    H2: el desempeño y el nivel de interacción permiten clasificar a los
        estudiantes en niveles de riesgo académico (alerta temprana).
    """

    PASS_THRESHOLD = None  # se fija dinámicamente (mediana) en assemble()

    def build_assessment_features(self, ad: pd.DataFrame) -> pd.DataFrame:
        ad = ad.copy()
        exam = ad[ad["assessment_type"] == "EXAM"].copy()
        exam["w_score"] = exam["score"] * exam["weight"]
        exam_agg = exam.groupby("guid_student_id").agg(
            weighted_score=("w_score", "sum"),
            total_weight=("weight", "sum"),
            n_exam=("guid_assess_id", "count"),
        ).reset_index()
        exam_agg["weighted_score"] = np.where(
            exam_agg["total_weight"] > 0,
            exam_agg["weighted_score"] / exam_agg["total_weight"], 0.0)

        overall_agg = ad.groupby("guid_student_id").agg(
            n_assessments=("guid_assess_id", "count"),
            n_finished=("status", lambda s: (s == "finished").sum()),
            mean_score_raw=("score", "mean"),
        ).reset_index()
        overall_agg["submission_rate"] = (
            overall_agg["n_finished"] / overall_agg["n_assessments"].replace(0, np.nan)
        ).fillna(0.0)

        agg = overall_agg.merge(
            exam_agg[["guid_student_id", "weighted_score", "n_exam"]],
            on="guid_student_id", how="left")
        agg["weighted_score"] = agg["weighted_score"].fillna(0.0)
        agg["n_exam"] = agg["n_exam"].fillna(0)
        return agg

    def build_vle_features(self, vle: pd.DataFrame) -> pd.DataFrame:
        return vle.groupby("guid_student_id").agg(
            total_clicks=("sum_clics", "sum"),
            mean_clicks=("sum_clics", "mean"),
            active_sessions=("sum_clics", "count"),
            active_weeks=("week1", "nunique"),
            n_resource_types=("type_assign", "nunique"),
        ).reset_index()

    def assemble(self, si, assess_feats, vle_feats) -> pd.DataFrame:
        df = si.merge(assess_feats, on="guid_student_id", how="left")
        df = df.merge(vle_feats, on="guid_student_id", how="left")
        for col in ["weighted_score", "n_assessments", "n_finished", "n_exam",
                    "mean_score_raw", "submission_rate"]:
            df[col] = df[col].fillna(0.0)
        for col in ["total_clicks", "mean_clicks", "active_sessions",
                    "active_weeks", "n_resource_types"]:
            df[col] = df[col].fillna(0.0)

        threshold = df["weighted_score"].median()
        self.PASS_THRESHOLD = threshold
        df["y_pass"] = (df["weighted_score"] >= threshold).astype(int)

        z_score = (df["weighted_score"] - df["weighted_score"].mean()) / df["weighted_score"].std(ddof=0)
        z_clicks = (df["total_clicks"] - df["total_clicks"].mean()) / df["total_clicks"].std(ddof=0)
        composite = 0.5 * z_score.fillna(0) + 0.5 * z_clicks.fillna(0)
        df["risk_composite"] = composite
        df["y_risk_ordinal"] = pd.qcut(composite, q=3, labels=[0, 1, 2]).astype(int)

        df["y_score_reg"] = df["weighted_score"]
        return df


fe = FeatureEngineer()
assess_feats = fe.build_assessment_features(ad_clean)
vle_feats = fe.build_vle_features(vle_clean)
full_df = fe.assemble(si_clean, assess_feats, vle_feats)

print("Umbral de aprobación (mediana):", round(fe.PASS_THRESHOLD, 3))
print(full_df["y_pass"].value_counts())
print(full_df["y_risk_ordinal"].value_counts())
full_df.head()


### 4.1 Visualizaciones EDA (univariado, bivariado, correlación, boxplot)

In [ ]:
num_cols = ["weighted_score", "total_clicks", "submission_rate",
            "num_of_prev_attempts", "studied_credits", "active_weeks"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, num_cols):
    sns.histplot(full_df[col], kde=True, ax=ax, color="#2b6cb0")
    ax.set_title(f"Distribución de {col}")
fig.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, col in zip(axes, ["weighted_score", "total_clicks", "submission_rate"]):
    sns.boxplot(y=full_df[col], ax=ax, color="#63b3ed")
    ax.set_title(f"Boxplot: {col}")
fig.tight_layout()
plt.show()

kurt_skew = pd.DataFrame({
    "variable": num_cols,
    "media": [full_df[c].mean() for c in num_cols],
    "desv_std": [full_df[c].std() for c in num_cols],
    "asimetria": [full_df[c].skew() for c in num_cols],
    "curtosis": [full_df[c].kurtosis() for c in num_cols],
})
kurt_skew


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(x="y_pass", y="total_clicks", data=full_df, ax=axes[0],
            palette=["#e53e3e", "#38a169"])
axes[0].set_title("Clics totales según Aprobado/No aprobado")
axes[0].set_xticklabels(["No aprobado", "Aprobado"])

sns.scatterplot(x="total_clicks", y="weighted_score", hue="y_pass",
                 data=full_df, ax=axes[1], palette=["#e53e3e", "#38a169"])
axes[1].set_title("Dispersión: Clics VLE vs. Desempeño (score)")
fig.tight_layout()
plt.show()


In [ ]:
corr_cols = ["weighted_score", "total_clicks", "mean_clicks", "active_sessions",
             "active_weeks", "n_resource_types", "submission_rate",
             "n_assessments", "num_of_prev_attempts", "studied_credits",
             "imd_band_ord", "age_band_ord"]
corr = full_df[corr_cols].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax,
            annot_kws={"size": 7})
ax.set_title("Matriz de correlación")
fig.tight_layout()
plt.show()

print("Correlación weighted_score vs total_clicks (evidencia de H1):",
      round(corr.loc["weighted_score", "total_clicks"], 3))


---
## 5. MODEL — Modelos predictivos supervisados y no supervisados

Se entrenan **3 algoritmos supervisados por cada tipo de variable objetivo**
(dicotómica, ordinal e intervalo/razón), más un modelo no supervisado
(K-Means) para explorar tendencias de agrupamiento.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                               GradientBoostingClassifier, GradientBoostingRegressor)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVR
from sklearn.cluster import KMeans
from sklearn.metrics import (precision_score, recall_score, f1_score, accuracy_score,
                              roc_auc_score, confusion_matrix, mean_squared_error, r2_score)


@dataclass
class ModelResult:
    """TAD para encapsular el resultado de un modelo entrenado."""
    name: str
    target_type: str
    y_test: np.ndarray
    y_pred: np.ndarray
    y_proba: Optional[np.ndarray] = None
    metrics: Dict[str, float] = field(default_factory=dict)
    feature_importances: Optional[Dict[str, float]] = None


class MetricsCalculator:
    """Cálculo manual de TP/FP/TN/FN y F1, más métricas estándar de sklearn."""

    @staticmethod
    def confusion_counts_binary(y_true, y_pred) -> Dict[str, int]:
        tp = int(np.sum((y_true == 1) & (y_pred == 1)))
        tn = int(np.sum((y_true == 0) & (y_pred == 0)))
        fp = int(np.sum((y_true == 0) & (y_pred == 1)))
        fn = int(np.sum((y_true == 1) & (y_pred == 0)))
        return {"TP": tp, "TN": tn, "FP": fp, "FN": fn}

    @staticmethod
    def manual_f1(tp, fp, fn):
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
        return precision, recall, f1

    def classification_metrics(self, y_test, y_pred, y_proba=None):
        out = {
            "accuracy": accuracy_score(y_test, y_pred),
            "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
            "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
            "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
        }
        if y_proba is not None and len(np.unique(y_test)) == 2:
            try:
                out["roc_auc"] = roc_auc_score(y_test, y_proba)
            except Exception:
                out["roc_auc"] = np.nan
        else:
            out["roc_auc"] = np.nan
        if len(np.unique(y_test)) == 2:
            cc = self.confusion_counts_binary(np.array(y_test), np.array(y_pred))
            p_m, r_m, f1_m = self.manual_f1(cc["TP"], cc["FP"], cc["FN"])
            out.update({"TP": cc["TP"], "FP": cc["FP"], "TN": cc["TN"], "FN": cc["FN"],
                        "precision_manual": p_m, "recall_manual": r_m, "f1_manual": f1_m})
        return out

    def regression_metrics(self, y_test, y_pred):
        return {"mse": mean_squared_error(y_test, y_pred), "r2": r2_score(y_test, y_pred)}


In [ ]:
class ModelPipeline:
    """Orquesta el entrenamiento de modelos supervisados (3 tipos de target)
    y un modelo no supervisado exploratorio (KMeans)."""

    FEATURES = ["num_of_prev_attempts", "studied_credits", "imd_band_ord", "age_band_ord",
                "total_clicks", "mean_clicks", "active_sessions", "active_weeks",
                "n_resource_types", "submission_rate", "n_assessments"]

    def __init__(self, df, pass_threshold):
        self.df = df
        self.pass_threshold = pass_threshold
        self.metrics_calc = MetricsCalculator()
        self.results = []

    def _Xy(self, target_col):
        return self.df[self.FEATURES].values, self.df[target_col].values

    def run_dichotomous(self):
        X, y = self._Xy("y_pass")
        X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
            X, y, self.df.index, test_size=0.3, random_state=RANDOM_STATE, stratify=y)
        scaler = StandardScaler().fit(X_train)
        X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)

        models = {
            "LogisticRegression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
            "RandomForestClassifier": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
            "KNN": KNeighborsClassifier(n_neighbors=5),
        }
        for name, model in models.items():
            model.fit(X_train_s, y_train)
            y_pred = model.predict(X_test_s)
            y_proba = model.predict_proba(X_test_s)[:, 1] if hasattr(model, "predict_proba") else None
            metrics = self.metrics_calc.classification_metrics(y_test, y_pred, y_proba)
            fi = dict(zip(self.FEATURES, model.feature_importances_)) if hasattr(model, "feature_importances_") else None
            self.results.append(ModelResult(name, "dicotomica_y_pass", y_test, y_pred, y_proba, metrics, fi))
        self._dicho_test_idx = idx_test

    def run_ordinal(self):
        X, y = self._Xy("y_risk_ordinal")
        X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
            X, y, self.df.index, test_size=0.3, random_state=RANDOM_STATE, stratify=y)
        scaler = StandardScaler().fit(X_train)
        X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)

        models = {
            "LogisticRegression_multinom": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
            "RandomForestClassifier_ord": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
            "GradientBoosting_ord": GradientBoostingClassifier(random_state=RANDOM_STATE),
        }
        for name, model in models.items():
            model.fit(X_train_s, y_train)
            y_pred = model.predict(X_test_s)
            metrics = self.metrics_calc.classification_metrics(y_test, y_pred, None)
            fi = dict(zip(self.FEATURES, model.feature_importances_)) if hasattr(model, "feature_importances_") else None
            self.results.append(ModelResult(name, "ordinal_y_risk", y_test, y_pred, None, metrics, fi))
        self._ord_test_idx = idx_test

    def run_regression(self):
        X, y = self._Xy("y_score_reg")
        X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
            X, y, self.df.index, test_size=0.3, random_state=RANDOM_STATE)
        scaler = StandardScaler().fit(X_train)
        X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)

        models = {
            "LinearRegression": LinearRegression(),
            "RandomForestRegressor": RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE),
            "SVR": SVR(kernel="rbf"),
        }
        y_research = self.df.loc[idx_test, "y_pass"].values
        for name, model in models.items():
            model.fit(X_train_s, y_train)
            y_pred = model.predict(X_test_s)
            metrics = self.metrics_calc.regression_metrics(y_test, y_pred)
            y_research_pred = (y_pred >= self.pass_threshold).astype(int)
            metrics["msePI2"] = mean_squared_error(y_research, y_research_pred)
            metrics["r2PI2"] = r2_score(y_research, y_research_pred)
            fi = dict(zip(self.FEATURES, model.feature_importances_)) if hasattr(model, "feature_importances_") else None
            self.results.append(ModelResult(name, "regresion_y_score", y_test, y_pred, None, metrics, fi))
        self._reg_test_idx = idx_test
        self._y_research = y_research

    def run_unsupervised(self, k=3):
        X = self.df[["weighted_score", "total_clicks", "submission_rate"]].values
        Xs = StandardScaler().fit_transform(X)
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(Xs)
        self.df["cluster"] = km.labels_
        return km

    def run_all(self):
        self.run_dichotomous()
        self.run_ordinal()
        self.run_regression()
        self.run_unsupervised()
        return self.results


pipeline = ModelPipeline(full_df, pass_threshold=fe.PASS_THRESHOLD)
results = pipeline.run_all()
for r in results:
    print("====", r.name, "|", r.target_type)
    print(r.metrics)


---
## 6. Evaluación de modelos — matrices de confusión, ROC, regresión



In [ ]:
dicho_results = [r for r in results if r.target_type == "dicotomica_y_pass"]
fig, axes = plt.subplots(1, len(dicho_results), figsize=(5 * len(dicho_results), 4))
for ax, r in zip(axes, dicho_results):
    cm = confusion_matrix(r.y_test, r.y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["No aprob.", "Aprob."], yticklabels=["No aprob.", "Aprob."])
    ax.set_title(r.name)
    ax.set_xlabel("Predicho"); ax.set_ylabel("Real")
fig.suptitle("Matrices de confusión — Modelo dicotómico")
fig.tight_layout()
plt.show()

pd.DataFrame([{**{"modelo": r.name}, **r.metrics} for r in dicho_results])


### 6.2 Variable ordinal (Riesgo académico: Alto / Medio / Bajo)

In [ ]:
ord_results = [r for r in results if r.target_type == "ordinal_y_risk"]
labels_ord = ["Alto riesgo", "Riesgo medio", "Bajo riesgo"]
fig, axes = plt.subplots(1, len(ord_results), figsize=(5 * len(ord_results), 4))
for ax, r in zip(axes, ord_results):
    cm = confusion_matrix(r.y_test, r.y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", ax=ax,
                xticklabels=labels_ord, yticklabels=labels_ord)
    ax.set_title(r.name, fontsize=9)
    ax.set_xlabel("Predicho"); ax.set_ylabel("Real")
fig.suptitle("Matrices de confusión — Modelo ordinal")
fig.tight_layout()
plt.show()

pd.DataFrame([{**{"modelo": r.name}, **r.metrics} for r in ord_results])


### 6.3 Variable de intervalo/razón (regresión sobre weighted_score)

In [ ]:
reg_results = [r for r in results if r.target_type == "regresion_y_score"]
fig, axes = plt.subplots(1, len(reg_results), figsize=(5 * len(reg_results), 4.5))
for ax, r in zip(axes, reg_results):
    ax.scatter(r.y_test, r.y_pred, alpha=0.7, color="#2b6cb0")
    lims = [min(r.y_test.min(), r.y_pred.min()), max(r.y_test.max(), r.y_pred.max())]
    ax.plot(lims, lims, "--", color="gray")
    ax.set_xlabel("y_test (real)"); ax.set_ylabel("y_pred (predicho)")
    ax.set_title(f"{r.name}\nMSE={r.metrics['mse']:.2f}  R2={r.metrics['r2']:.2f}")
fig.suptitle("Regresión: predicción del puntaje ponderado")
fig.tight_layout()
plt.show()

pd.DataFrame([{**{"modelo": r.name}, **r.metrics} for r in reg_results])


### 6.4 Importancia de variables (Random Forest — modelo dicotómico)

In [ ]:
rf_dicho = [r for r in results if r.name == "RandomForestClassifier"][0]
fi = pd.Series(rf_dicho.feature_importances).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 5))
fi.plot(kind="barh", ax=ax, color="#2f855a")
ax.invert_yaxis()
ax.set_title("Importancia de variables — Random Forest (Aprobado/No aprobado)")
fig.tight_layout()
plt.show()
fi


### 6.5 Exploración no supervisada (K-Means, k=3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
sc = ax.scatter(full_df["total_clicks"], full_df["weighted_score"],
                 c=full_df["cluster"], cmap="viridis", s=50, alpha=0.8)
ax.set_xlabel("Clics totales en el VLE"); ax.set_ylabel("Puntaje ponderado (score)")
ax.set_title("Segmentación no supervisada (K-Means, k=3)")
legend1 = ax.legend(*sc.legend_elements(), title="Cluster")
ax.add_artist(legend1)
fig.tight_layout()
plt.show()


---
## 7. Exportación de entregables (CSV, métricas caso a caso, esquema SQL)


In [ ]:
import os
os.makedirs("outputs", exist_ok=True)

# --- Predicciones caso a caso (y_test, y_pred) por modelo ---
rows = []
for r in results:
    for i, (yt, yp) in enumerate(zip(r.y_test, r.y_pred)):
        rows.append({"modelo": r.name, "tipo_variable": r.target_type,
                     "caso_idx": i, "y_test": yt, "y_pred": yp})
pred_df = pd.DataFrame(rows)
pred_df.to_csv("outputs/predicciones_caso_a_caso.csv", index=False)

# --- Métricas generales por modelo ---
metrics_rows = []
for r in results:
    m = {"modelo": r.name, "tipo_variable": r.target_type}
    m.update(r.metrics)
    metrics_rows.append(m)
metrics_df = pd.DataFrame(metrics_rows)
metrics_df.to_csv("outputs/metricas_generales.csv", index=False)

# --- Dataset final procesado ---
full_df.to_csv("outputs/dataset_procesado.csv", index=False)

print("Archivos CSV exportados en /outputs")
metrics_df


In [ ]:
# --- Esquema RDBMS reproducible (SQLite) ---
import sqlite3, gzip, shutil

db_path = "outputs/oulad_kongo.db"
if os.path.exists(db_path):
    os.remove(db_path)

xls_raw = pd.ExcelFile(DATA_PATH)
conn = sqlite3.connect(db_path)
table_map = {
    "StudentInfo": "student_info", "Registration": "registration",
    "Assesss_detail": "assessment_detail", "Assess Plan": "assessment_plan",
    "VLE_clickStream": "vle_clickstream", "Vle_modules": "vle_modules",
    "cursos": "cursos",
}
for sheet, table in table_map.items():
    pd.read_excel(xls_raw, sheet_name=sheet).to_sql(table, conn, if_exists="replace", index=False)

cur = conn.cursor()
cur.execute("CREATE INDEX IF NOT EXISTS idx_si_student ON student_info(guid_student_id);")
cur.execute("CREATE INDEX IF NOT EXISTS idx_ad_student ON assessment_detail(guid_student_id);")
cur.execute("CREATE INDEX IF NOT EXISTS idx_vle_student ON vle_clickstream(guid_student_id);")
conn.commit()

sql_dump = "outputs/oulad_kongo_schema.sql"
with open(sql_dump, "w", encoding="utf-8") as f:
    for line in conn.iterdump():
        f.write(f"{line}\n")
conn.close()

with open(sql_dump, "rb") as f_in, gzip.open(sql_dump + ".gz", "wb") as f_out:
    shutil.copyfileobj(f_in, f_out)

print("Esquema SQL generado:", sql_dump, "y su versión comprimida .sql.gz")
